# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² mass spectrometry dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is defined by a [Croissant schema](https://mlcommons.org/croissant/) and accessible via the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is available
!pip install -q mlcroissant

## 1. Data Loading
Load dataset metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print core metadata information
meta = dataset.metadata
print(f"{meta.name}\n{'-'*len(meta.name)}\n{meta.description}")

## 2. Data Overview
Review available record sets and their corresponding fields/columns.

All references are made using each entity's `@id` as specified in the Croissant schema.

In [ ]:
# List all RecordSets in the dataset with their `@id` and other details
pp = pprint.PrettyPrinter(indent=2)
record_sets = []
print("Available Record Sets:")
for rs in meta.record_sets:
    print(f"- Name: {rs.name}\n  @id: {rs.id}\n  Description: {rs.description}")
    record_sets.append(rs.id)
    # List fields/columns for each record set, using their @id
    print("  Fields/Columns:")
    for f in rs.fields:
        print(f"    - {f.name} (@id: {f.id}) [type: {f.data_type}] {('[column: ' + f.column.id + ']' if f.column else '')}")
    print()


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

Reference all record sets and fields by their `@id` for clarity and reproducibility.

In [ ]:
# Extract all data from each RecordSet
dataframes = {}
for rs_id in record_sets:
    print(f'Loading records from RecordSet: {rs_id}')
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records. Columns: {dataframes[rs_id].columns.tolist()}")

# For demonstration, pick the first available RecordSet for further analysis
main_record_set = record_sets[0]
df = dataframes[main_record_set]
print(f'\nSample rows from {main_record_set}:')
df.head()

## 4. Exploratory Data Analysis (EDA)
Common data processing steps: filtering, normalization, and grouping by key attributes.

All fields selected are referenced by their `@id`.

In [ ]:
# Identify a numeric field and a categorical group field by @id
# These should match the fields listed in the overview above!
# For example purposes, let's often expect these kinds of ids from proteomics tables:

# Example field IDs (please adjust to the actual field @id's from the overview if needed)
numeric_field_id = None
group_field_id = None

# Dynamically select the first numeric field (Float or Integer) and a group field (e.g., 'description' or class)
from mlcroissant._src.structure_fields.data_types import DataType
rs_obj = [rs for rs in meta.record_sets if rs.id == main_record_set][0]
for field in rs_obj.fields:
    if field.data_type in [DataType.FLOAT, DataType.INTEGER] and numeric_field_id is None:
        numeric_field_id = field.id
    if (field.data_type == DataType.TEXT or field.data_type == DataType.INTEGER) and group_field_id is None and 'desc' in field.name.lower():
        group_field_id = field.id
    # Otherwise pick the first text field as group label
    if group_field_id is None and field.data_type == DataType.TEXT:
        group_field_id = field.id
if numeric_field_id is None:
    print('No suitable numeric field (@id) detected for analysis.')
else:
    print(f'Numeric field selected: {numeric_field_id}')
if group_field_id is None:
    print('No suitable group field (@id) detected for grouping.')
else:
    print(f'Group field selected: {group_field_id}')

# Filter, normalize, and group if possible
if numeric_field_id in df.columns:
    # Attempt numeric conversion and filtering
    col = df[numeric_field_id]
    # Try convert to numeric, coerce errors to NaN
    col_numeric = pd.to_numeric(col, errors='coerce')
    threshold = col_numeric.mean()   # Use mean as threshold for demonstration
    filtered_df = df[col_numeric > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} records")
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (col_numeric - col_numeric.mean()) / col_numeric.std()
    print(f"Normalized {numeric_field_id} (showing head):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group by categorical field if available
    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped {numeric_field_id} mean by {group_field_id} (first 5):")
        display(grouped_df.head())
else:
    print(f"The field {numeric_field_id} was not found in the DataFrame columns.")

## 5. Visualization
Visualize data distributions or relationships between fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If a numeric and group field is selected, plot distribution
if numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
    
    # Boxplot by group if available
    if group_field_id in df.columns:
        plt.figure(figsize=(12,4))
        sns.boxplot(x=df[group_field_id], y=pd.to_numeric(df[numeric_field_id], errors='coerce'))
        plt.title(f'{numeric_field_id} distribution by {group_field_id}')
        plt.xticks(rotation=30, ha='right')
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.tight_layout()
        plt.show()
else:
    print(f"No numeric field available in DataFrame for plotting.")

## 6. Conclusion

- Loaded dataset metadata and explored available record sets and fields using their Croissant `@id` references.
- Parsed the main record set into a DataFrame and demonstrated basic filtering, normalization, and grouping using referenced fields.
- Visualized key numeric distributions.

This workflow can be extended for in-depth biological or statistical protein analyses by further leveraging field `@id` structure and domain-specific post-processing.